# Davinci — Long-Context Curriculum & Evaluation

This notebook defines a staged long-context experiment plan and evaluation harness for Davinci. It does **not** claim that changing a context-length config creates long-context capability. Every stage requires continued training, memory profiling, and retrieval/position-sensitive evaluation.

**Critical compatibility note:** Notebook 5 currently uses dense attention masks and recomputes the full prefix. It is a correctness reference for short sequences, not a 500K-capable attention implementation. Do not run long-sequence training with that implementation. First integrate and validate a memory-efficient sliding-window/global attention kernel and KV-cache policy.

## 1. Runtime and experiment configuration

In [ ]:
import os, json, math, random, time, hashlib
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
print("torch",torch.__version__,"device",DEVICE)
if DEVICE=="cuda":
    p=torch.cuda.get_device_properties(0)
    print(p.name,"VRAM GiB",round(p.total_memory/2**30,2))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
EXPERIMENT_DIR=Path("/content/drive/MyDrive/Davinci/long_context")
EXPERIMENT_DIR.mkdir(parents=True,exist_ok=True)

# Curriculum stages are proposed gates, not automatic promises.
CURRICULUM=[
 {"context":4096,"min_steps":1000,"gate":"baseline"},
 {"context":8192,"min_steps":1000,"gate":"pass_4k_and_8k_eval"},
 {"context":16384,"min_steps":1500,"gate":"pass_8k_and_16k_eval"},
 {"context":32768,"min_steps":2000,"gate":"pass_16k_and_32k_eval"},
 {"context":65536,"min_steps":2500,"gate":"pass_32k_and_64k_eval"},
 {"context":131072,"min_steps":3000,"gate":"pass_64k_and_128k_eval"},
 {"context":262144,"min_steps":4000,"gate":"pass_128k_and_256k_eval"},
 {"context":500000,"min_steps":5000,"gate":"pass_256k_and_500k_eval"}
]
LOCAL_WINDOW=4096
GLOBAL_EVERY_N_LAYERS=4
print(json.dumps(CURRICULUM,indent=2))

## 2. Establish the actual attention implementation status

The current reference implementation materializes a dense `T × T` mask. Its memory grows quadratically. This guard makes it difficult to accidentally run a large-context test with the reference implementation.

In [ ]:
REFERENCE_DENSE_ATTENTION=True
MAX_REFERENCE_TEST_TOKENS=2048
def assert_attention_backend_safe(sequence_length, backend_is_memory_efficient):
    if sequence_length>MAX_REFERENCE_TEST_TOKENS and not backend_is_memory_efficient:
        raise RuntimeError(
          f"Refusing {sequence_length:,}-token run with dense reference attention. "
          "Integrate a verified memory-efficient local/global attention backend first."
        )
assert_attention_backend_safe(1024,backend_is_memory_efficient=False)
try:
    assert_attention_backend_safe(8192,backend_is_memory_efficient=False)
except RuntimeError as e:
    print("Expected safety guard:",e)

## 3. RoPE extension experiment design

Do not choose a RoPE scaling method based on a config name alone. Compare unscaled RoPE, linear position interpolation, and a documented NTK-aware/dynamic variant using identical checkpoints and evaluation sets. The code below provides a simple linear interpolation position transform for controlled experiments; it is not a recommendation that this method will work best.

In [ ]:
def rope_position_ids(seq_len,device,position_scale=1.0):
    """Linear position interpolation: effective position = position * scale."""
    if position_scale<=0: raise ValueError("position_scale must be positive")
    return torch.arange(seq_len,device=device,dtype=torch.float32)*position_scale

def rope_cos_sin(seq_len,head_dim,theta=10000.0,position_scale=1.0,device="cpu"):
    assert head_dim%2==0
    pos=rope_position_ids(seq_len,device,position_scale)
    inv=1.0/(theta**(torch.arange(0,head_dim,2,device=device,dtype=torch.float32)/head_dim))
    phase=torch.outer(pos,inv)
    return phase.cos(),phase.sin()

def compare_rope_scales(seq_len=128,head_dim=64,theta=10000.0):
    results={}
    for name,scale in [("unscaled",1.0),("2x_interpolation",0.5),("4x_interpolation",0.25)]:
        c,s=rope_cos_sin(seq_len,head_dim,theta,scale,DEVICE)
        results[name]={"cos_shape":list(c.shape),"sin_shape":list(s.shape),
                       "finite":bool(torch.isfinite(c).all() and torch.isfinite(s).all()),
                       "last_position_phase_norm":float(torch.sqrt(c[-1].pow(2)+s[-1].pow(2)).mean())}
    return results
print(compare_rope_scales())

## 4. Attention mask semantics and KV-cache accounting

For a hybrid schedule where every fourth layer is global, this estimates retained KV entries if local layers retain only the last `LOCAL_WINDOW` tokens and global layers retain the full prefix. It is a theoretical estimate; the actual kernel/runtime must enforce this eviction policy.

In [ ]:
def estimate_kv_cache_bytes(seq_len,batch_size,num_layers,num_kv_heads,head_dim,dtype_bytes=2,
                              local_window=4096,global_every=4):
    global_layers=num_layers//global_every
    local_layers=num_layers-global_layers
    retained=global_layers*seq_len+local_layers*min(seq_len,local_window)
    return 2*batch_size*num_kv_heads*head_dim*dtype_bytes*retained

def gib(n): return n/(1024**3)
cache_rows=[]
for stage in CURRICULUM:
    n=stage["context"]
    size=estimate_kv_cache_bytes(n,1,28,4,64,2,LOCAL_WINDOW,GLOBAL_EVERY_N_LAYERS)
    cache_rows.append({"context":n,"kv_cache_GiB_per_sequence":round(gib(size),3)})
cache_rows

## 5. Create synthetic long-context retrieval examples

This generator creates deterministic needle-in-a-haystack examples. It is useful for testing whether the evaluation pipeline can place and retrieve evidence at different positions. Synthetic examples are not a substitute for natural long-document benchmarks, multi-hop reasoning, or tool-use evaluation.

In [ ]:
def make_needle_example(num_fillers,needle_position,seed=0):
    rng=random.Random(seed)
    key=f"DAVINCI_KEY_{seed:08d}"
    value=f"DAVINCI_VALUE_{seed:08d}"
    needle=f"Record {key} has the unique value {value}."
    fillers=[f"Document {i}: "+("background information about systems, language, and data. "*8)
             for i in range(num_fillers)]
    pos=max(0,min(len(fillers),int(needle_position*len(fillers))))
    fillers.insert(pos,needle)
    prompt="\n".join(fillers)+f"\nQuestion: What is the unique value associated with {key}?\nAnswer:"
    return {"prompt":prompt,"answer":value,"key":key,"needle_fraction":needle_position}

def build_eval_suite(count=30):
    fractions=[0.02,0.1,0.25,0.5,0.75,0.9,0.98]
    rows=[]
    for i in range(count):
        frac=fractions[i%len(fractions)]
        rows.append(make_needle_example(50+i*2,frac,SEED+i))
    return rows
suite=build_eval_suite()
print("Examples:",len(suite),"sample target:",suite[0]["answer"],"fraction:",suite[0]["needle_fraction"])

## 6. Tokenize and filter examples by target context

Set the tokenizer path from Notebook 3. The notebook records actual token lengths rather than assuming character counts map to token counts. For large target lengths, synthetic examples must be expanded substantially and then audited for position distribution.

In [ ]:
TOKENIZER_MODEL="/content/drive/MyDrive/Davinci/tokenizer/davinci_unigram_32000.model"
import sentencepiece as spm
assert Path(TOKENIZER_MODEL).is_file(),TOKENIZER_MODEL
sp=spm.SentencePieceProcessor(model_file=TOKENIZER_MODEL)
def tokenize_eval_example(ex):
    ids=sp.encode(ex["prompt"],out_type=int)
    answer_ids=sp.encode(ex["answer"],out_type=int)
    return {**ex,"input_ids":ids,"answer_ids":answer_ids,"token_length":len(ids)}
tokenized=[tokenize_eval_example(x) for x in suite]
print("Token length range:",min(x["token_length"] for x in tokenized),max(x["token_length"] for x in tokenized))

## 7. Retrieval evaluation adapter

Connect a trained model and generation implementation here. The adapter must return generated answer text. Do not use the reference Notebook 5 model for large contexts. Record exact model checkpoint, tokenizer, context length, backend, and generation parameters with every run.

In [ ]:
def model_answer_adapter(example,model,generate_fn,max_new_tokens=24):
    # generate_fn(model, input_ids, max_new_tokens) must return token IDs.
    input_ids=torch.tensor([example["input_ids"]],device=DEVICE)
    with torch.no_grad():
        generated=generate_fn(model,input_ids,max_new_tokens=max_new_tokens)
    if isinstance(generated,torch.Tensor):
        generated=generated[0].tolist()
    # If generate_fn returns full prompt + completion, decode only suffix.
    suffix=generated[len(example["input_ids"]):] if len(generated)>=len(example["input_ids"]) else generated
    return sp.decode(suffix).strip()

def score_exact_retrieval(examples,answer_fn):
    rows=[]
    for ex in examples:
        pred=answer_fn(ex)
        rows.append({"token_length":ex["token_length"],"needle_fraction":ex["needle_fraction"],
                     "expected":ex["answer"],"prediction":pred,
                     "correct":ex["answer"] in pred})
    return {"accuracy":sum(r["correct"] for r in rows)/max(1,len(rows)),"rows":rows}

# Example usage after implementing generation:
# results=score_exact_retrieval(tokenized,lambda ex:model_answer_adapter(ex,model,generate_fn))
# print("Exact retrieval accuracy:",results["accuracy"])

## 8. Stage gate policy

A stage advances only when all checks pass. Thresholds below are starting policy values and must be calibrated against a baseline model and task difficulty. Require no catastrophic regression on shorter contexts.

In [ ]:
GATE_POLICY={
 "min_retrieval_accuracy":0.90,
 "max_short_context_accuracy_drop":0.03,
 "max_validation_loss_regression_fraction":0.05,
 "require_no_nan":True,
 "require_peak_memory_headroom_fraction":0.10
}
def stage_gate(metrics,policy=GATE_POLICY):
    checks={
      "retrieval":metrics.get("retrieval_accuracy",0)>=policy["min_retrieval_accuracy"],
      "short_context_regression":metrics.get("short_context_accuracy_drop",1)<=policy["max_short_context_accuracy_drop"],
      "validation_loss":metrics.get("validation_loss_regression_fraction",1)<=policy["max_validation_loss_regression_fraction"],
      "finite":metrics.get("all_finite",False),
      "memory_headroom":metrics.get("memory_headroom_fraction",0)>=policy["require_peak_memory_headroom_fraction"]
    }
    return {"passed":all(checks.values()),"checks":checks}
print("Gate policy initialized:",GATE_POLICY)

## 9. Experiment manifest and results ledger

In [ ]:
def save_stage_result(stage_context,metrics,checkpoint,backend,rope_method,notes=""):
    record={
      "timestamp_utc":time.strftime("%Y-%m-%dT%H:%M:%SZ",time.gmtime()),
      "context_length":stage_context,"metrics":metrics,"checkpoint":str(checkpoint),
      "attention_backend":backend,"rope_method":rope_method,"notes":notes,
      "gate":stage_gate(metrics)
    }
    path=EXPERIMENT_DIR/f"stage_{stage_context}_result.json"
    path.write_text(json.dumps(record,indent=2),encoding="utf-8")
    return record
print("Experiment directory:",EXPERIMENT_DIR)

## 10. Long-context benchmark checklist

Evaluate each stage on:
- Needle retrieval at beginning, middle, and end.
- Multiple needles and conflicting distractors.
- Multi-hop retrieval across separated documents.
- Long-document summarization with coverage checks.
- Question answering where evidence is absent (abstention).
- Code repository questions with distant definitions/usages.
- Tool histories where the relevant observation is far from the final query.
- Standardized external suites (for example LongBench, RULER, InfiniteBench), after checking their licenses and task formats.

For each test, record tokenized length, evidence positions, exact checkpoint, decoding settings, and confidence/abstention behavior.

## 11. Required engineering work before 128K+

1. Replace dense attention with a proven implementation supporting local sliding windows and periodic global layers; test numerical parity on short sequences.
2. Implement correct KV-cache storage and local-layer eviction, plus global-layer retention.
3. Profile training activations and cache memory at every stage; include optimizer state and fragmentation.
4. Choose and validate positional extension (RoPE scaling or another position scheme) empirically.
5. Add gradient checkpointing and memory-efficient training kernels.
6. Create naturally distributed long-document training data, not only synthetic needles.
7. Run ablations: all-global vs hybrid; different local windows; different global-layer frequencies; RoPE methods.
8. Only attempt 256K/500K after 128K retrieval and quality gates pass repeatedly.

**No model weights are modified in this notebook.** It defines the curriculum, accounting, synthetic evaluation data, and gate framework; actual continued training is a separate run that must use the optimized model implementation.